# 01. CSE Banking Data Collection

**Tickers**: COMB.N0000.LK, HNB.N0000.LK, NDB.N0000.LK, BFLN.N0000.LK, DFCC.N0000.LK, SAMP.N0000.LK

**Sources**: CSE API → Twelve Data → Alpha Vantage → Sample Data (fallback)  
**Cloud**: AWS S3 upload

**Status**: 🔄 Implementing CSE API Integration

In [ ]:
import sys
import os
sys.path.append('..')  # Add parent directory to path

from cse_api_client import CSEDataClient
import pandas as pd
from datetime import datetime
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Sri Lankan Banking Tickers
BANKING_TICKERS = [
    'COMB.N0000.LK',  # Commercial Bank of Ceylon
    'HNB.N0000.LK',   # Hatton National Bank
    'NDB.N0000.LK',   # National Development Bank
    'DFCC.N0000.LK',  # DFCC Bank
    'SAMP.N0000.LK',  # Sampath Bank
    'BFLN.N0000.LK'   # Bank of Ceylon
]

def main():
    """Main data collection function"""
    print("🚀 Starting CSE Banking Data Collection")
    print("=" * 50)

    # Initialize CSE API client
    client = CSEDataClient()

    # Date range: 2 years back for sufficient historical data
    end_date = datetime.now().strftime('%Y-%m-%d')
    start_date = (datetime.now() - pd.DateOffset(years=2)).strftime('%Y-%m-%d')

    print(f"📅 Date Range: {start_date} to {end_date}")
    print(f"🏦 Tickers: {', '.join(BANKING_TICKERS)}")
    print()

    # Fetch data for all tickers
    print("📡 Fetching data from CSE API with fallbacks...")
    results = client.fetch_multiple_tickers(BANKING_TICKERS, start_date, end_date)

    if not results:
        print("❌ No data could be fetched from any source")
        return

    # Combine all ticker data into multi-index DataFrame (yfinance format)
    print("🔄 Processing and combining data...")
    combined_data = {}
    summary_stats = []

    for ticker, df in results.items():
        if df is not None and not df.empty:
            # Validate data quality
            if client.validate_data(df, ticker):
                combined_data[ticker] = df

                # Calculate summary statistics
                latest_price = df['Close'].iloc[-1] if not df.empty else 0
                avg_volume = df['Volume'].mean() if not df.empty else 0
                data_points = len(df)

                summary_stats.append({
                    'Ticker': ticker,
                    'Latest Price': f"${latest_price:.2f}",
                    'Data Points': data_points,
                    'Avg Volume': f"{avg_volume:,.0f}",
                    'Date Range': f"{df.index.min().date()} to {df.index.max().date()}"
                })

                print(f"✅ {ticker}: {data_points} days, latest: ${latest_price:.2f}")
            else:
                print(f"❌ {ticker}: Data validation failed")
        else:
            print(f"❌ {ticker}: No data available")

    if not combined_data:
        print("❌ No valid data to save")
        return

    # Create multi-index DataFrame (compatible with existing analysis)
    try:
        final_df = pd.concat(combined_data, axis=1, keys=combined_data.keys())
        print(f"\n📊 Combined data shape: {final_df.shape}")

        # Save to CSV
        output_path = '../data/banking_ohlcv_raw.csv'
        final_df.to_csv(output_path)
        print(f"💾 Saved to {output_path}")

        # Display summary table
        print("\n📈 DATA COLLECTION SUMMARY")
        print("=" * 60)
        summary_df = pd.DataFrame(summary_stats)
        print(summary_df.to_string(index=False))

        print(f"\n✅ Successfully collected data for {len(combined_data)}/{len(BANKING_TICKERS)} tickers")
        print("🎯 Ready for preprocessing and analysis!")

    except Exception as e:
        print(f"❌ Error processing final data: {e}")
        # Save individual ticker files as fallback
        for ticker, df in combined_data.items():
            try:
                df.to_csv(f'../data/{ticker.replace(".N0000.LK", "")}_data.csv')
                print(f"💾 Saved individual file: {ticker}")
            except Exception as e2:
                print(f"❌ Failed to save {ticker}: {e2}")

if __name__ == "__main__":
    main()

: 